# Tool Use 프로젝트: 개인 커리어 어시스턴트

이번 노트북에서는 OpenAI의 **Tool Use(함수 호출)** 기능을 활용하여 개인 커리어 어시스턴트 챗봇을 구축합니다. LLM이 외부 함수를 호출하고 그 결과를 활용하는 전체 흐름을 실습합니다.

## 개요

| 주제 | 내용 |
|------|------|
| Tool Use 개념 | LLM이 함수를 호출하는 원리 |
| Tool 함수 정의 | Python 함수와 JSON 스키마 작성 |
| Tool 호출 핸들링 | IF문 vs 동적 디스패치 비교 |
| 챗봇 루프 | finish_reason 기반 반복 처리 |
| Gradio UI | 대화형 인터페이스 구축 |

## 학습 목표

1. Tool Use(Function Calling)의 동작 원리 이해하기
2. Tool 함수 정의와 JSON 스키마 작성법 익히기
3. Tool 호출 핸들링 패턴 비교하기
4. Gradio를 활용한 챗봇 UI 구축하기

---

## 프로젝트 소개

이 프로젝트에서는 **자기 자신을 소개하는 AI 챗봇**을 만듭니다. 챗봇은:
- 여러분의 이력과 경험에 대한 질문에 답변합니다
- 관심 있는 사용자의 연락처를 기록합니다
- 답변할 수 없는 질문을 로깅합니다
- Gradio UI로 웹에서 접근할 수 있습니다

---

## 1. Tool Use란?

**Tool Use(Function Calling)**는 LLM이 직접 외부 함수를 호출할 수 있는 기능입니다. LLM은 텍스트만 생성할 수 있지만, Tool Use를 통해 실제 세상과 상호작용할 수 있습니다.

### Tool Use 전체 흐름

```
┌──────────────────────────────────────────────────────────────────────────┐
│                         Tool Use 처리 흐름                              │
├──────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│  ┌────────┐     ┌────────┐     ┌──────────────┐     ┌────────────┐     │
│  │ 사용자 │────▶│  LLM   │────▶│ Tool 호출    │────▶│ 함수 실행  │     │
│  │ 메시지 │     │  판단  │     │ 판단         │     │ (Python)   │     │
│  └────────┘     └────────┘     └──────────────┘     └─────┬──────┘     │
│                                                           │             │
│  ┌────────┐     ┌────────┐     ┌──────────────┐          │             │
│  │ 최종   │◀────│  LLM   │◀────│ 결과 반환    │◀─────────┘             │
│  │ 응답   │     │  종합  │     │ (tool result)│                        │
│  └────────┘     └────────┘     └──────────────┘                        │
│                                                                         │
└──────────────────────────────────────────────────────────────────────────┘
```

### 핵심 포인트

1. **LLM은 함수를 직접 실행하지 않습니다** — 어떤 함수를 어떤 인자로 호출할지 "제안"할 뿐입니다
2. **실제 실행은 우리 코드에서 처리합니다** — LLM의 제안을 받아 Python으로 함수를 실행합니다
3. **결과를 다시 LLM에 전달합니다** — LLM이 함수 결과를 참고하여 최종 응답을 생성합니다

---

## 2. 환경 설정

In [30]:
# 환경 설정 및 라이브러리 임포트
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

load_dotenv(override=True)

api_key = os.getenv('OPENAI_API_KEY')
if api_key:
    print("API key found.")
else:
    print("No API key was found")

MODEL = "gpt-4o-mini"
client = OpenAI()

API key found.


---

## 3. 알림 시스템 설정

실제 서비스에서는 중요한 이벤트가 발생했을 때 알림을 받는 것이 유용합니다. 여기서는 **print 기반 로깅**을 기본으로 사용하고, 필요시 Pushover 같은 푸시 알림 서비스를 연동할 수 있도록 구성합니다.

> **선택사항**: [Pushover](https://pushover.net/)를 사용하면 실제 모바일 푸시 알림을 받을 수 있습니다. `.env` 파일에 `PUSHOVER_USER`와 `PUSHOVER_TOKEN`을 설정하세요.

In [42]:
# 알림 함수 정의
# 기본: print 기반 로깅 / 선택: Pushover 푸시 알림

pushover_user = os.getenv('PUSHOVER_USER', '')
pushover_token = os.getenv('PUSHOVER_TOKEN', '')
pushover_enabled = bool(pushover_user and pushover_token)

def push(message):
    """알림 전송 함수 — Pushover가 설정되면 푸시 알림, 아니면 print 로깅"""
    print(f"[알림] {message}")
    if pushover_enabled:
        try:
            import requests
            response = requests.post(
                "https://api.pushover.net/1/messages.json",
                data={"user": pushover_user, "token": pushover_token, "message": message}
            )
            print(f"  → Pushover 응답: HTTP {response.status_code} / {response.text}")
        except Exception as e:
            print(f"  → Pushover 전송 실패: {e}")

if pushover_enabled:
    print(f"Pushover 설정 확인 — USER: {pushover_user[:6]}..., TOKEN: {pushover_token[:6]}...")
    push("Pushover 연결 테스트")
else:
    print("Pushover 미설정 — print 기반 로깅을 사용합니다.")

Pushover 설정 확인 — USER: uwskvm..., TOKEN: ab7351...
[알림] Pushover 연결 테스트
  → Pushover 응답: HTTP 200 / {"status":1,"request":"784bdf5b-a85e-4279-be8d-f0898931a241"}


In [46]:
push("hello123")

[알림] hello123
  → Pushover 응답: HTTP 200 / {"status":1,"request":"df19b6b8-352d-4149-84e1-83fb221cced2"}


---

## 4. Tool 함수 정의

Tool Use에서는 **두 가지**를 정의해야 합니다:

```
┌─────────────────────────────────────────────────────────────────┐
│                   Tool 구성 요소                                │
├─────────────────────────────┬───────────────────────────────────┤
│     Python 함수             │       JSON 스키마                 │
├─────────────────────────────┼───────────────────────────────────┤
│  • 실제 비즈니스 로직 수행   │  • LLM에게 함수 정보 전달         │
│  • 데이터베이스 저장         │  • 함수명, 설명, 파라미터 정의    │
│  • API 호출                 │  • 필수/선택 파라미터 구분         │
│  • 알림 전송                │  • 타입과 설명으로 LLM 가이드     │
└─────────────────────────────┴───────────────────────────────────┘
```

### 우리가 만들 Tool 함수

| 함수명 | 역할 | 사용 시점 |
|--------|------|----------|
| `record_user_details` | 관심 있는 사용자 정보 기록 | 사용자가 연락처를 남기고 싶을 때 |
| `record_unknown_question` | 답변 불가 질문 기록 | 프로필에 없는 정보를 질문받았을 때 |

In [19]:
# Tool 함수 1: 사용자 정보 기록

def record_user_details(email, name="이름 미제공", notes=""):
    """관심 있는 사용자의 연락처 정보를 기록합니다."""
    push(f"새로운 관심 사용자! 이름: {name}, 이메일: {email}, 메모: {notes}")
    return {"status": "success", "message": f"{name}님의 정보가 기록되었습니다."}


# Tool 함수 2: 답변 불가 질문 기록

def record_unknown_question(question):
    """챗봇이 답변할 수 없는 질문을 기록합니다."""
    push(f"답변 불가 질문: {question}")
    return {"status": "logged", "message": "질문이 기록되었습니다. 추후 답변을 준비하겠습니다."}

### JSON 스키마 정의

LLM이 Tool을 올바르게 호출하려면 **JSON 스키마**로 함수의 구조를 알려줘야 합니다. 스키마에는 함수명, 설명, 파라미터의 타입과 설명이 포함됩니다.

> **중요**: `description` 필드가 상세할수록 LLM이 적절한 시점에 Tool을 호출합니다. 이것은 **도구를 위한 프롬프트 엔지니어링**입니다.

In [21]:
# Tool JSON 스키마 정의

record_user_details_json = {
    "type": "function",
    "function": {
        "name": "record_user_details",
        "description": "사용자가 연락처 정보(이메일 등)를 제공하거나, 연락 받기를 원하거나, 관심을 표현할 때 이 함수를 호출하세요. 사용자의 이름, 이메일, 관련 메모를 기록합니다.",
        "parameters": {
            "type": "object",
            "properties": {
                "email": {
                    "type": "string",
                    "description": "사용자의 이메일 주소"
                },
                "name": {
                    "type": "string",
                    "description": "사용자의 이름 (선택)"
                },
                "notes": {
                    "type": "string",
                    "description": "추가 메모 — 관심 분야, 대화 맥락 등"
                }
            },
            "required": ["email"],
            "additionalProperties": False
        }
    }
}

record_unknown_question_json = {
    "type": "function",
    "function": {
        "name": "record_unknown_question",
        "description": "프로필 정보에 없는 내용을 질문받아 답변할 수 없을 때 이 함수를 호출하세요. 답변하지 못한 질문을 기록하여 추후 대응할 수 있도록 합니다.",
        "parameters": {
            "type": "object",
            "properties": {
                "question": {
                    "type": "string",
                    "description": "답변할 수 없는 질문의 내용"
                }
            },
            "required": ["question"],
            "additionalProperties": False
        }
    }
}

tools = [record_user_details_json, record_unknown_question_json]

# 스키마 확인
print(json.dumps(tools, indent=2, ensure_ascii=False))

[
  {
    "type": "function",
    "function": {
      "name": "record_user_details",
      "description": "사용자가 연락처 정보(이메일 등)를 제공하거나, 연락 받기를 원하거나, 관심을 표현할 때 이 함수를 호출하세요. 사용자의 이름, 이메일, 관련 메모를 기록합니다.",
      "parameters": {
        "type": "object",
        "properties": {
          "email": {
            "type": "string",
            "description": "사용자의 이메일 주소"
          },
          "name": {
            "type": "string",
            "description": "사용자의 이름 (선택)"
          },
          "notes": {
            "type": "string",
            "description": "추가 메모 — 관심 분야, 대화 맥락 등"
          }
        },
        "required": [
          "email"
        ],
        "additionalProperties": false
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "record_unknown_question",
      "description": "프로필 정보에 없는 내용을 질문받아 답변할 수 없을 때 이 함수를 호출하세요. 답변하지 못한 질문을 기록하여 추후 대응할 수 있도록 합니다.",
      "parameters": {
        "type": "object",
        "properties": {
          "question": 

---

## 5. Tool 호출 핸들링

LLM이 Tool 호출을 요청하면, 우리 코드에서 해당 함수를 실행해야 합니다. 두 가지 방식을 비교해보겠습니다.

### 방식 비교

| 방식 | 장점 | 단점 |
|------|------|------|
| **IF문 분기** | 명시적, 이해하기 쉬움 | Tool이 많아지면 코드가 길어짐 |
| **`globals()` 동적 디스패치** | 확장성 좋음, 코드 간결 | 보안 주의 필요, 디버깅 약간 어려움 |

```python
# 방식 1: IF문 분기
if tool_name == "record_user_details":
    result = record_user_details(**arguments)
elif tool_name == "record_unknown_question":
    result = record_unknown_question(**arguments)

# 방식 2: globals() 동적 디스패치
tool = globals().get(tool_name)
result = tool(**arguments)
```

이 프로젝트에서는 **동적 디스패치 방식**을 채택합니다. Tool을 추가할 때 핸들러 코드를 수정할 필요가 없기 때문입니다.

In [47]:
# Tool 호출 핸들러 — globals() 기반 동적 디스패치

def handle_tool_calls(tool_calls):
    """LLM의 Tool 호출 요청을 처리하고 결과를 반환합니다."""
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"  [Tool 호출] {tool_name}({arguments})")
        
        # globals()에서 함수를 동적으로 찾아 실행
        tool = globals().get(tool_name)
        if tool:
            result = tool(**arguments)
        else:
            result = {"error": f"알 수 없는 Tool: {tool_name}"}
        
        results.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": json.dumps(result, ensure_ascii=False)
        })
    return results

---

## 6. 프로필 데이터 준비

챗봇이 여러분에 대해 답변하려면 **프로필 정보**가 필요합니다. 여기서는 샘플 데이터를 직접 정의합니다.

> **실습 팁**: 아래 샘플 데이터를 여러분의 실제 정보로 교체하세요! LinkedIn PDF를 읽어오거나, 직접 텍스트를 작성할 수 있습니다.

```python
# PDF에서 프로필 읽기 (선택사항)
# from pypdf import PdfReader
# reader = PdfReader("my_resume.pdf")
# linkedin = "".join(page.extract_text() for page in reader.pages)
```

In [48]:
# 샘플 프로필 데이터 — 자신의 정보로 교체하세요!

name = "김철수"  # 여러분의 이름으로 변경

# LinkedIn 프로필 또는 이력서 내용
linkedin = """김철수
시니어 소프트웨어 엔지니어 | AI/ML 전문가

경력:
- ABC테크 (2020-현재): AI 플랫폼 개발 리드
  - LLM 기반 고객 서비스 자동화 시스템 구축
  - RAG 파이프라인 설계 및 운영
  - 팀 규모: 5명 → 12명 성장 리드

- XYZ소프트 (2017-2020): 백엔드 개발자
  - Python/FastAPI 기반 마이크로서비스 아키텍처 구축
  - 실시간 데이터 파이프라인 개발

학력:
- 서울대학교 컴퓨터공학과 석사 (2017)
- 서울대학교 컴퓨터공학과 학사 (2015)

기술 스택:
Python, PyTorch, LangChain, FastAPI, PostgreSQL, Docker, AWS

관심 분야:
LLM 에이전트, RAG 시스템, MLOps"""

# 추가 요약 정보
summary = """김철수는 AI와 소프트웨어 엔지니어링에 열정적인 개발자입니다.
현재 LLM 기반 에이전트 시스템 개발에 집중하고 있으며,
실제 비즈니스 문제를 AI로 해결하는 것에 관심이 있습니다.
기술 커뮤니티 활동과 오픈소스 기여를 즐기며,
새로운 기술을 배우고 공유하는 것을 좋아합니다."""

print(f"프로필 이름: {name}")
print(f"LinkedIn 데이터 길이: {len(linkedin)} 문자")
print(f"요약 데이터 길이: {len(summary)} 문자")

프로필 이름: 김철수
LinkedIn 데이터 길이: 390 문자
요약 데이터 길이: 154 문자


---

## 7. 시스템 프롬프트 설계

시스템 프롬프트는 챗봇의 **성격과 행동 규칙**을 정의합니다. 좋은 시스템 프롬프트에는 다음이 포함됩니다:

| 구성 요소 | 역할 |
|-----------|------|
| 역할 정의 | 챗봇이 누구인지, 무엇을 하는지 |
| 컨텍스트 | 참고할 프로필/이력 정보 |
| 행동 규칙 | Tool 사용 조건, 답변 스타일 |
| 톤 앤 매너 | 친절하고 전문적인 태도 |

In [49]:
# 시스템 프롬프트 구성

system_prompt = f"""당신은 {name}을 대신하여 대화하는 AI 어시스턴트입니다.
당신의 역할은 {name}의 경력, 기술, 경험에 대한 질문에 친절하고 전문적으로 답변하는 것입니다.

## 참고 정보

### 이력/프로필
{linkedin}

### 추가 요약
{summary}

## 행동 규칙

1. 위 정보를 기반으로 {name}에 대한 질문에 성실히 답변하세요.
2. 답변은 항상 긍정적이고 전문적인 톤을 유지하세요.
3. 사용자가 연락처(이메일 등)를 제공하면 record_user_details 함수를 호출하세요.
4. 프로필에 없는 정보를 질문받으면 record_unknown_question 함수를 호출한 후, 모른다고 솔직히 답변하세요.
5. 한국어로 답변하세요.
"""

print("시스템 프롬프트 길이:", len(system_prompt), "문자")
print("\n--- 시스템 프롬프트 미리보기 ---")
print(system_prompt[:500] + "...")

시스템 프롬프트 길이: 887 문자

--- 시스템 프롬프트 미리보기 ---
당신은 김철수을 대신하여 대화하는 AI 어시스턴트입니다.
당신의 역할은 김철수의 경력, 기술, 경험에 대한 질문에 친절하고 전문적으로 답변하는 것입니다.

## 참고 정보

### 이력/프로필
김철수
시니어 소프트웨어 엔지니어 | AI/ML 전문가

경력:
- ABC테크 (2020-현재): AI 플랫폼 개발 리드
  - LLM 기반 고객 서비스 자동화 시스템 구축
  - RAG 파이프라인 설계 및 운영
  - 팀 규모: 5명 → 12명 성장 리드

- XYZ소프트 (2017-2020): 백엔드 개발자
  - Python/FastAPI 기반 마이크로서비스 아키텍처 구축
  - 실시간 데이터 파이프라인 개발

학력:
- 서울대학교 컴퓨터공학과 석사 (2017)
- 서울대학교 컴퓨터공학과 학사 (2015)

기술 스택:
Python, PyTorch, LangChain, FastAPI, PostgreSQL, Docker, AWS

관심 분야:
LLM 에이전트, RAG 시스템, MLOps

...


---

## 8. 챗봇 루프 구현

Tool Use를 활용하는 챗봇의 핵심은 **while 루프**입니다. LLM의 응답에서 `finish_reason`을 확인하여:

- `"tool_calls"` → Tool을 실행하고 결과를 메시지에 추가한 후 다시 LLM 호출
- `"stop"` → 최종 응답이므로 루프 종료

```
┌──────────────────────────────────────────────────────────────┐
│                    챗봇 루프 구조                             │
│                                                              │
│  사용자 메시지 추가                                          │
│         │                                                    │
│         ▼                                                    │
│  ┌─────────────────┐                                        │
│  │  LLM API 호출   │◀──────────────────────┐                │
│  └────────┬────────┘                       │                │
│           │                                │                │
│     finish_reason?                         │                │
│      ┌────┴─────┐                          │                │
│      ▼          ▼                          │                │
│  "stop"    "tool_calls"                    │                │
│    │           │                           │                │
│    │     Tool 실행 + 결과 추가 ─────────────┘                │
│    ▼                                                        │
│  최종 응답 반환                                              │
└──────────────────────────────────────────────────────────────┘
```

In [50]:
# 챗봇 함수 구현

def chat(message, history):
    """Gradio 챗봇 인터페이스 함수"""
    # 메시지 구성
    messages = [{"role": "system", "content": system_prompt}]
    
    # 대화 히스토리 추가
    for h in history:
        messages.append({"role": "user", "content": h["content"]}) if h["role"] == "user" else \
        messages.append({"role": "assistant", "content": h["content"]})
    
    # 현재 메시지 추가
    messages.append({"role": "user", "content": message})
    
    # Tool Use 루프
    while True:
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools
        )
        
        choice = response.choices[0]
        
        # Tool 호출이 필요한 경우
        if choice.finish_reason == "tool_calls":
            # assistant 메시지 추가 (tool_calls 포함)
            messages.append(choice.message)
            # Tool 실행 및 결과 추가
            tool_results = handle_tool_calls(choice.message.tool_calls)
            messages.extend(tool_results)
        else:
            # 최종 응답
            return choice.message.content

# 테스트 실행
test_response = chat("안녕하세요, 자기소개 부탁드립니다.", [])
print(test_response)

안녕하세요! 저는 김철수입니다. 현재 ABC테크에서 시니어 소프트웨어 엔지니어로 근무하고 있으며, AI 및 머신러닝 분야에 전문성을 가지고 있습니다. LLM 기반의 고객 서비스 자동화 시스템을 개발하는 프로젝트를 리드하고 있으며, RAG 파이프라인 설계 및 운영을 담당하고 있습니다. 

이전에 XYZ소프트에서는 백엔드 개발자로서 Python/FastAPI를 활용한 마이크로서비스 아키텍처 구축과 실시간 데이터 파이프라인 개발 경험이 있습니다. 

제 학력은 서울대학교 컴퓨터공학과에서 석사(2017)와 학사(2015)를 마쳤습니다. 기술 스택으로는 Python, PyTorch, LangChain, FastAPI, PostgreSQL, Docker, AWS를 사용하고 있으며, LLM 에이전트, RAG 시스템, MLOps에도 많은 관심을 가지고 있습니다. 

AI를 통해 실제 비즈니스 문제를 해결하는 것에 열정을 가지고 있으며, 기술 커뮤니티 활동과 오픈소스 기여를 즐깁니다. 무엇이든 궁금하신 점이 있으면 말씀해 주세요!


In [ ]:
# Gradio UI 실행

gr.ChatInterface(
    fn=chat,
    title=f"{name} 커리어 어시스턴트",
    description=f"{name}의 경력과 기술에 대해 물어보세요!",
    examples=[
        "어떤 경력을 가지고 계신가요?",
        "주요 기술 스택은 무엇인가요?",
        "AI 관련 프로젝트 경험이 있나요?"
    ]
).launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


  [Tool 호출] record_unknown_question({'question': '특허 있어요?'})
[알림] 답변 불가 질문: 특허 있어요?
  → Pushover 응답: HTTP 200 / {"status":1,"request":"403fe0eb-362e-4952-aaf2-6a0c4b6571f8"}


---

## 9. 배포 가이드: HuggingFace Spaces

만든 챗봇을 **HuggingFace Spaces**에 배포하여 누구나 접근할 수 있게 할 수 있습니다.

### 배포 단계

1. **HuggingFace 계정 생성**: [huggingface.co](https://huggingface.co)에서 가입
2. **새 Space 생성**: `New Space` → SDK: `Gradio` 선택
3. **코드 업로드**: 아래 파일들을 Space에 업로드
   - `app.py` — 챗봇 코드 (이 노트북의 코드를 `.py` 파일로 변환)
   - `requirements.txt` — `openai`, `gradio` 등 의존성
4. **Secrets 설정**: Space Settings → Variables and Secrets에서 `OPENAI_API_KEY` 등록
5. **배포 확인**: Space가 빌드되면 자동으로 URL이 생성됩니다

### `requirements.txt` 예시

```
openai
gradio
python-dotenv
```

### 주의사항

- API 키는 **절대 코드에 하드코딩하지 마세요** — 반드시 Secrets 사용
- 무료 Space는 리소스가 제한적이므로 가벼운 모델(`gpt-4o-mini`)을 권장합니다
- 프로필 데이터를 코드에 포함하거나 별도 파일로 업로드하세요

---

## 10. 연습 과제 및 확장 아이디어

### 연습 과제

1. **프로필 교체**: 샘플 데이터를 자신의 실제 정보로 교체하고 챗봇을 테스트해보세요
2. **Tool 추가**: 새로운 Tool 함수를 만들어보세요
   - 예: `schedule_meeting` — 미팅 일정 요청 기록
   - 예: `get_portfolio` — 포트폴리오 링크 반환
3. **HuggingFace Spaces 배포**: 실제로 배포해보고 URL을 공유해보세요

### 확장 아이디어

| 아이디어 | 설명 |
|----------|------|
| PDF 이력서 연동 | `pypdf`로 PDF에서 프로필 자동 추출 |
| RAG 추가 | 프로필 정보를 벡터 DB에 저장하고 검색 |
| 다국어 지원 | 한국어/영어 자동 전환 |
| 데이터베이스 연동 | SQLite로 사용자 정보 영구 저장 |
| 이메일 알림 | Pushover 대신 이메일 알림 시스템 |

---

## 참고 자료

- [OpenAI Function Calling 문서](https://platform.openai.com/docs/guides/function-calling)
- [Gradio 공식 문서](https://www.gradio.app/docs)
- [HuggingFace Spaces 가이드](https://huggingface.co/docs/hub/spaces)